In [ ]:
# Cell 1 — Dependencies
# Run once per environment. Safe to re-run (pip is a no-op when up-to-date).
#
# NOTE: every package below lives on *public* PyPI. We pin --index-url to
# https://pypi.org/simple/ explicitly so this cell keeps working even when
# the environment's default index is a private mirror with an expired token
# (e.g. AWS CodeArtifact, Artifactory, Nexus). To force the env's default
# index instead, set USE_DEFAULT_PIP_INDEX = True below.
USE_DEFAULT_PIP_INDEX = False

import sys, subprocess

_PKGS = [
    "opencv-python>=4.8", "opencv-contrib-python>=4.8",
    "mediapipe>=0.10",
    # numpy capped at <2.0 to coexist with SageMaker's pre-installed
    # faceformer / mlpytils / pyarrow / numba (all require numpy<2).
    "numpy>=1.23,<2.0", "scipy>=1.10",
    "tqdm>=4.66", "requests>=2.31",
    "boto3>=1.34", "smart_open[s3]>=7.0",
]
_cmd = [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade"]
if not USE_DEFAULT_PIP_INDEX:
    # --index-url completely overrides whatever pip.conf / PIP_INDEX_URL is set
    # in the environment, so an expired CodeArtifact token can't break us.
    _cmd += ["--index-url", "https://pypi.org/simple/", "--retries", "3"]
_cmd += _PKGS
print("$", " ".join(_cmd))
subprocess.check_call(_cmd)

# If you'd rather refresh the private mirror's token, run (outside the notebook):
#   aws codeartifact login --tool pip --domain elai-ml \
#       --domain-owner 477103407489 --repository pypi-store --region us-east-2
# then set USE_DEFAULT_PIP_INDEX = True and re-run this cell.


In [ ]:
# Cell 2 — Imports & global logging
from __future__ import annotations

import hashlib
import logging
import os
import random
import shutil
import subprocess
import sys
import urllib.parse
from collections import deque
from dataclasses import dataclass
from pathlib import Path
from typing import Deque, Dict, Generator, List, Optional, Tuple

import cv2
import numpy as np
import requests
from tqdm.auto import tqdm

import boto3
from botocore.config import Config as BotoConfig
from botocore.exceptions import ClientError

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
LOG = logging.getLogger("seamless_extender")
LOG.info("OpenCV %s · DIS optical flow: %s · MediaPipe ready",
         cv2.__version__, hasattr(cv2, "DISOpticalFlow_create"))


In [ ]:
# Cell 3 — Configuration
#
# Edit the values below or override them via environment variables before
# launching Jupyter (the notebook reads from os.environ first).

# ---------- Input / output (S3 or presigned HTTP) ----------
INPUT_S3_URL: str = os.environ.get(
    "INPUT_S3_URL",
    "s3://my-bucket/raw_clips/host_30s.mp4",     # or "https://…?X-Amz-…"
)
OUTPUT_S3_FOLDER: str = os.environ.get(
    "OUTPUT_S3_FOLDER",
    "s3://my-bucket/extended/",                  # folder URI or full key
)

# ---------- AWS / S3-compatible credentials ----------
# Leave any of these as None to use the default boto3 chain
# (IAM role, ~/.aws/credentials, AWS_* env vars).
# Set S3_ENDPOINT_URL only for non-AWS providers
# (MinIO, Yandex Object Storage, Backblaze B2, Cloudflare R2, …).
AWS_ACCESS_KEY_ID: Optional[str]    = os.environ.get("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY: Optional[str] = os.environ.get("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN: Optional[str]    = os.environ.get("AWS_SESSION_TOKEN")
AWS_REGION: Optional[str]           = os.environ.get("AWS_REGION", "us-east-1")
S3_ENDPOINT_URL: Optional[str]      = os.environ.get("S3_ENDPOINT_URL") or None

# ---------- Local working folders ----------
CACHE_ROOT   = Path(os.environ.get("SEAMLESS_CACHE_DIR", ".seamless_cache"))
DOWNLOAD_DIR = CACHE_ROOT / "downloads"   # deduplicated input downloads
RENDER_DIR   = CACHE_ROOT / "renders"     # mp4 outputs before upload
for _d in (CACHE_ROOT, DOWNLOAD_DIR, RENDER_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# ---------- Algorithm parameters ----------
TARGET_DURATION_SEC: float = 300.0   # target length of the extended video
SIMILARITY_THRESHOLD: float = 0.05   # normalized distance ceiling for jumps
MIN_GAP_FRAMES: int       = 150      # min |i - j| between transition endpoints
JUMP_PROBABILITY: float   = 0.30     # per-eligible-frame jump probability
TRANSITION_FRAMES: int    = 5        # length of motion-compensated cross-fade
COOLDOWN_FRAMES: int      = 90       # min linear frames between consecutive jumps
VELOCITY_ALIGN: float     = 0.6      # cosine threshold for velocity-direction match
RANDOM_SEED: Optional[int] = 42      # set None for a non-deterministic walk

# ---------- Cleanup ----------
DELETE_LOCAL_VIDEO_AFTER_UPLOAD: bool = True  # keep only .npz / .npy caches

print("Input :", INPUT_S3_URL)
print("Output:", OUTPUT_S3_FOLDER)
print("Cache :", CACHE_ROOT.resolve())


In [ ]:
# Cell 4 — S3 download / upload with deduplicating local cache
#
# Public functions:
#   download_from_s3(url, local_path=None) -> Path
#   upload_to_s3(local_path, s3_destination) -> str
#   cleanup_local_videos(*paths) -> None

def _build_s3_client():
    """Create a boto3 S3 client honoring configured creds + endpoint URL.

    Works against AWS as well as MinIO / Yandex Cloud / R2 / Backblaze when
    `S3_ENDPOINT_URL` is set.
    """
    session_kwargs: Dict[str, Optional[str]] = {"region_name": AWS_REGION}
    if AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY:
        session_kwargs.update(
            aws_access_key_id=AWS_ACCESS_KEY_ID,
            aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
        )
        if AWS_SESSION_TOKEN:
            session_kwargs["aws_session_token"] = AWS_SESSION_TOKEN
    session = boto3.session.Session(
        **{k: v for k, v in session_kwargs.items() if v}
    )
    client_kwargs: Dict[str, object] = {
        "config": BotoConfig(retries={"max_attempts": 5, "mode": "adaptive"}),
    }
    if S3_ENDPOINT_URL:
        client_kwargs["endpoint_url"] = S3_ENDPOINT_URL
    return session.client("s3", **client_kwargs)


def _parse_s3_uri(uri: str) -> Tuple[str, str]:
    """`s3://bucket/key/path` → (bucket, key)."""
    parsed = urllib.parse.urlparse(uri)
    if parsed.scheme != "s3" or not parsed.netloc:
        raise ValueError(f"Not an s3:// URI: {uri!r}")
    return parsed.netloc, parsed.path.lstrip("/")


def _cache_path_for_url(url: str) -> Path:
    """Deterministic, content-addressed cache filename for an input URL.

    The hash is computed over the *URL string* (presigned tokens included is
    fine — re-running with the exact same URL means we already have the file).
    The original extension is preserved so OpenCV/FFmpeg are happy.
    """
    digest = hashlib.sha1(url.encode("utf-8")).hexdigest()[:16]
    url_path = urllib.parse.urlparse(url).path
    ext = Path(url_path).suffix or ".mp4"
    if len(ext) > 6 or "?" in ext:   # query-string artifact: fall back
        ext = ".mp4"
    return DOWNLOAD_DIR / f"{digest}{ext}"


def download_from_s3(url: str, local_path: Optional[Path] = None) -> Path:
    """Download `url` to a local cached file; reuse when already present.

    Supports:
      * `s3://bucket/key`           — via ``boto3.client('s3').download_file``
      * `http(s)://…` (presigned)   — via ``requests`` with chunked streaming
    """
    if local_path is None:
        local_path = _cache_path_for_url(url)
    local_path = Path(local_path)

    if local_path.exists() and local_path.stat().st_size > 0:
        LOG.info("Cache hit (%.1f MB) → %s",
                 local_path.stat().st_size / 1e6, local_path)
        return local_path

    local_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = local_path.with_suffix(local_path.suffix + ".part")
    scheme = urllib.parse.urlparse(url).scheme.lower()

    if scheme in ("http", "https"):
        LOG.info("HTTP download: %s", url.split("?", 1)[0])
        with requests.get(url, stream=True, timeout=120) as resp:
            resp.raise_for_status()
            total = int(resp.headers.get("Content-Length", 0)) or None
            with open(tmp_path, "wb") as fh, tqdm(
                total=total, unit="B", unit_scale=True, unit_divisor=1024,
                desc="Downloading", leave=False,
            ) as bar:
                for chunk in resp.iter_content(chunk_size=1024 * 1024):
                    if not chunk:
                        continue
                    fh.write(chunk)
                    bar.update(len(chunk))
    elif scheme == "s3":
        bucket, key = _parse_s3_uri(url)
        LOG.info("S3 download: s3://%s/%s", bucket, key)
        s3 = _build_s3_client()
        try:
            head = s3.head_object(Bucket=bucket, Key=key)
            total = int(head.get("ContentLength", 0)) or None
        except ClientError as exc:
            raise FileNotFoundError(f"S3 object missing: {url}") from exc
        with tqdm(total=total, unit="B", unit_scale=True, unit_divisor=1024,
                  desc="Downloading", leave=False) as bar:
            s3.download_file(
                bucket, key, str(tmp_path),
                Callback=lambda n: bar.update(n),
            )
    else:
        raise ValueError(f"Unsupported URL scheme {scheme!r}: {url!r}")

    tmp_path.rename(local_path)
    LOG.info("Downloaded → %s (%.1f MB)",
             local_path, local_path.stat().st_size / 1e6)
    return local_path


def upload_to_s3(local_path: Path, s3_destination: str) -> str:
    """Upload `local_path` to S3 with a tqdm progress bar.

    `s3_destination` may be a full ``s3://bucket/key`` URI or a "folder"
    URI like ``s3://bucket/extended/`` — in the latter case the basename
    of `local_path` is appended automatically. Multipart upload kicks in
    transparently for files larger than 8 MB.
    """
    local_path = Path(local_path)
    if not local_path.is_file():
        raise FileNotFoundError(local_path)
    if not s3_destination.startswith("s3://"):
        raise ValueError("s3_destination must be an s3:// URI")

    bucket, key = _parse_s3_uri(s3_destination)
    if not key or key.endswith("/"):
        key = f"{key}{local_path.name}" if key else local_path.name

    total = local_path.stat().st_size
    s3 = _build_s3_client()
    LOG.info("S3 upload: s3://%s/%s (%.1f MB)", bucket, key, total / 1e6)
    extra = {"ContentType": "video/mp4"} if local_path.suffix.lower() == ".mp4" else None
    with tqdm(total=total, unit="B", unit_scale=True, unit_divisor=1024,
              desc="Uploading", leave=False) as bar:
        s3.upload_file(
            str(local_path), bucket, key,
            ExtraArgs=extra,
            Callback=lambda n: bar.update(n),
        )
    final_uri = f"s3://{bucket}/{key}"
    LOG.info("Uploaded → %s", final_uri)
    return final_uri


def cleanup_local_videos(*paths: Path) -> None:
    """Remove heavy local video artifacts; .npz / .npy caches are preserved."""
    video_exts = {".mp4", ".mov", ".mkv", ".webm", ".avi", ".part"}
    for p in paths:
        if not p:
            continue
        p = Path(p)
        try:
            if p.is_file() and p.suffix.lower() in video_exts:
                size_mb = p.stat().st_size / 1e6
                p.unlink()
                LOG.info("Cleaned up local video: %s (%.1f MB)", p, size_mb)
        except OSError as exc:
            LOG.warning("Could not delete %s: %s", p, exc)


In [ ]:
# Cell 5 — Video metadata & MediaPipe feature extraction
# (with normalization against the shoulder midpoint + .npz caching)

# ---------- Landmark subsets ----------
POSE_INDICES: List[int] = [
    11, 12,            # shoulders (also the normalization anchor)
    13, 14,            # elbows
    15, 16,            # wrists
    17, 18,            # pinkies
    19, 20,            # index fingers
    21, 22,            # thumbs
]
FACE_INDICES: List[int] = [
    33, 133, 159, 145,          # left eye corners + upper/lower lid
    362, 263, 386, 374,         # right eye corners + upper/lower lid
    61, 291, 0, 17, 13, 14,     # outer & inner lip extremes
    78, 308,                    # inner mouth corners
    152, 10,                    # chin tip, forehead center
    234, 454,                   # left/right cheek
    1, 4, 5,                    # nose ridge
]
THUMB_SIZE: Tuple[int, int] = (96, 96)        # (w, h) for similarity thumbnails
FLOW_SIZE: Tuple[int, int]  = (320, 180)      # (w, h) for velocity estimation


# ---------- Data classes ----------
@dataclass
class VideoMetadata:
    path: str
    frame_count: int
    fps: float
    width: int
    height: int
    fourcc: str


@dataclass
class FrameFeatures:
    landmarks: np.ndarray      # (N, K, 3) normalized landmark coordinates
    valid_mask: np.ndarray     # (N,) True if pose landmarks detected
    thumb_gray: np.ndarray     # (N, H_t, W_t) uint8 center-crop thumbnails
    velocity_mag: np.ndarray   # (N,) mean magnitude of frame→next flow
    velocity_vec: np.ndarray   # (N, 2) mean vector of frame→next flow


# ---------- Helpers ----------
def video_cache_key(video_path: str) -> str:
    """Stable cache key from a video file's path + size + mtime."""
    p = Path(video_path).resolve()
    stat = p.stat()
    h = hashlib.sha1()
    h.update(p.as_posix().encode("utf-8"))
    h.update(str(stat.st_size).encode("utf-8"))
    h.update(str(int(stat.st_mtime)).encode("utf-8"))
    return h.hexdigest()[:16]


def get_video_metadata(path: str) -> VideoMetadata:
    """Probe a video file with OpenCV and return its properties."""
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Input video not found: {path}")
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise IOError(f"Could not open video: {path}")
    try:
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps         = float(cap.get(cv2.CAP_PROP_FPS)) or 30.0
        width       = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fourcc_int  = int(cap.get(cv2.CAP_PROP_FOURCC))
    finally:
        cap.release()
    fourcc = "".join(chr((fourcc_int >> (8 * i)) & 0xFF) for i in range(4)).strip()
    if frame_count <= 0:
        raise ValueError(f"Video reports zero frames: {path}")
    return VideoMetadata(
        path=path, frame_count=frame_count, fps=fps,
        width=width, height=height, fourcc=fourcc,
    )


def _make_dis_flow(preset: int = cv2.DISOPTICAL_FLOW_PRESET_MEDIUM):
    """Construct a DIS optical-flow calculator, gracefully falling back."""
    if hasattr(cv2, "DISOpticalFlow_create"):
        flow = cv2.DISOpticalFlow_create(preset)
        try:
            flow.setUseMeanNormalization(True)
            flow.setUseSpatialPropagation(True)
        except Exception:
            pass
        return flow
    LOG.warning("cv2.DISOpticalFlow_create unavailable; using Farneback.")
    return None


def _calc_flow(flow_calc, prev_gray: np.ndarray, cur_gray: np.ndarray) -> np.ndarray:
    """Compute dense optical flow with DIS or Farneback fallback."""
    if flow_calc is not None:
        return flow_calc.calc(prev_gray, cur_gray, None)
    return cv2.calcOpticalFlowFarneback(
        prev_gray, cur_gray, None,
        0.5, 3, 15, 3, 5, 1.2, 0,
    )


def _extract_landmarks_one(pose_results, face_results) -> Optional[np.ndarray]:
    """Convert MediaPipe results to a single normalized (K, 3) array.

    Returns ``None`` when pose landmarks are absent — those frames are
    flagged "non-loopable" downstream so they can't serve as either end
    of a transition.
    """
    if pose_results.pose_landmarks is None:
        return None
    pose_lms = pose_results.pose_landmarks.landmark
    l_sh, r_sh = pose_lms[11], pose_lms[12]
    cx = (l_sh.x + r_sh.x) * 0.5
    cy = (l_sh.y + r_sh.y) * 0.5
    scale = max(1e-6, float(np.hypot(l_sh.x - r_sh.x, l_sh.y - r_sh.y)))

    coords: List[List[float]] = []
    for idx in POSE_INDICES:
        lm = pose_lms[idx]
        coords.append([(lm.x - cx) / scale,
                       (lm.y - cy) / scale,
                       lm.z / scale])

    if face_results.multi_face_landmarks:
        face_lms = face_results.multi_face_landmarks[0].landmark
        for idx in FACE_INDICES:
            lm = face_lms[idx]
            coords.append([(lm.x - cx) / scale,
                           (lm.y - cy) / scale,
                           lm.z / scale])
    else:
        for _ in FACE_INDICES:
            coords.append([np.nan, np.nan, np.nan])
    return np.asarray(coords, dtype=np.float32)


def extract_features(meta: VideoMetadata,
                     cache_dir: Path,
                     cache_key: str) -> FrameFeatures:
    """Walk every frame, extract landmarks/thumbs/velocity, cache & return."""
    cache_file = cache_dir / f"{cache_key}_features.npz"
    if cache_file.is_file():
        LOG.info("Loading cached features from %s", cache_file)
        data = np.load(cache_file)
        return FrameFeatures(
            landmarks=data["landmarks"],
            valid_mask=data["valid_mask"],
            thumb_gray=data["thumb_gray"],
            velocity_mag=data["velocity_mag"],
            velocity_vec=data["velocity_vec"],
        )

    # Lazy + explicit sub-module imports. `mediapipe.solutions` is a
    # sub-package, not an attribute of the top-level module, and `import
    # mediapipe as mp` doesn't auto-bind it on every build (notably the
    # Linux wheels used by SageMaker). Importing the leaves directly
    # works on every released 0.10.x version.
    import mediapipe                                        # noqa: F401
    import mediapipe.solutions.pose as mp_pose
    import mediapipe.solutions.face_mesh as mp_face_mesh

    LOG.info("Extracting per-frame features from %s", meta.path)
    pose_solution = mp_pose.Pose(
        static_image_mode=False,
        model_complexity=1,
        smooth_landmarks=True,
        enable_segmentation=False,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    face_solution = mp_face_mesh.FaceMesh(
        static_image_mode=False,
        max_num_faces=1,
        refine_landmarks=False,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )

    n = meta.frame_count
    k = len(POSE_INDICES) + len(FACE_INDICES)
    landmarks    = np.full((n, k, 3), np.nan, dtype=np.float32)
    valid_mask   = np.zeros(n, dtype=bool)
    thumb_gray   = np.zeros((n, THUMB_SIZE[1], THUMB_SIZE[0]), dtype=np.uint8)
    velocity_mag = np.zeros(n, dtype=np.float32)
    velocity_vec = np.zeros((n, 2), dtype=np.float32)

    cap = cv2.VideoCapture(meta.path)
    if not cap.isOpened():
        raise IOError(f"Could not open video: {meta.path}")
    flow_calc = _make_dis_flow(cv2.DISOPTICAL_FLOW_PRESET_FAST)
    prev_gray_small: Optional[np.ndarray] = None
    read_n = n

    try:
        for i in tqdm(range(n), desc="Analyzing frames", unit="frame"):
            ret, frame_bgr = cap.read()
            if not ret:
                LOG.warning("Stream ended at frame %d (expected %d); truncating.",
                            i, n)
                read_n = i
                break

            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            pose_res = pose_solution.process(frame_rgb)
            face_res = face_solution.process(frame_rgb)

            lms = _extract_landmarks_one(pose_res, face_res)
            if lms is not None:
                landmarks[i] = lms
                pose_slice = lms[:len(POSE_INDICES)]
                valid_mask[i] = bool(np.isfinite(pose_slice).all())

            ch, cw = meta.height // 2, meta.width // 2
            y0, x0 = meta.height // 4, meta.width // 4
            center = frame_bgr[y0:y0 + ch, x0:x0 + cw]
            thumb_gray[i] = cv2.resize(
                cv2.cvtColor(center, cv2.COLOR_BGR2GRAY),
                THUMB_SIZE, interpolation=cv2.INTER_AREA,
            )

            gray_small = cv2.resize(
                cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY),
                FLOW_SIZE, interpolation=cv2.INTER_AREA,
            )
            if prev_gray_small is not None:
                flow = _calc_flow(flow_calc, prev_gray_small, gray_small)
                velocity_mag[i] = float(np.linalg.norm(flow, axis=2).mean())
                velocity_vec[i] = flow.mean(axis=(0, 1))
            prev_gray_small = gray_small
    finally:
        cap.release()
        pose_solution.close()
        face_solution.close()

    if read_n < n:
        landmarks    = landmarks[:read_n]
        valid_mask   = valid_mask[:read_n]
        thumb_gray   = thumb_gray[:read_n]
        velocity_mag = velocity_mag[:read_n]
        velocity_vec = velocity_vec[:read_n]
        meta.frame_count = read_n

    invalid = int((~valid_mask).sum())
    if invalid:
        LOG.warning("Pose not detected in %d/%d frames (non-loopable).",
                    invalid, valid_mask.size)

    cache_dir.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        cache_file,
        landmarks=landmarks, valid_mask=valid_mask,
        thumb_gray=thumb_gray, velocity_mag=velocity_mag,
        velocity_vec=velocity_vec,
    )
    LOG.info("Cached features → %s", cache_file)

    return FrameFeatures(
        landmarks=landmarks, valid_mask=valid_mask,
        thumb_gray=thumb_gray, velocity_mag=velocity_mag,
        velocity_vec=velocity_vec,
    )


In [ ]:
# Cell 6 — Distance matrix · transition graph · random-walk sequencer

@dataclass
class DistanceWeights:
    landmark: float = 1.0
    pixel: float    = 1.0
    velocity: float = 0.5


@dataclass
class WalkConfig:
    p_jump: float           = 0.30
    transition_len: int     = 5
    cooldown_frames: int    = 90
    recent_destinations: int = 8


@dataclass
class Op:
    """One step in the rendering plan: linear playback or a transition jump."""
    kind: str            # 'linear' or 'transition'
    src: int             # frame we are playing or jumping from
    dst: int = -1        # destination frame (only for 'transition')


# ---------- Pairwise distance helpers ----------
def _pairwise_l2(x: np.ndarray) -> np.ndarray:
    sq = np.sum(x * x, axis=1)
    g  = x @ x.T
    d2 = np.maximum(sq[:, None] + sq[None, :] - 2.0 * g, 0.0)
    return np.sqrt(d2).astype(np.float32)


def _pairwise_mse(x: np.ndarray) -> np.ndarray:
    sq  = np.sum(x * x, axis=1)
    g   = x @ x.T
    sse = np.maximum(sq[:, None] + sq[None, :] - 2.0 * g, 0.0)
    return (sse / float(x.shape[1])).astype(np.float32)


def _normalize_robust(m: np.ndarray) -> np.ndarray:
    finite = m[np.isfinite(m)]
    if finite.size == 0:
        return m
    ref = float(np.quantile(finite, 0.95))
    if ref <= 0:
        return m
    return np.clip(m / ref, 0.0, 1.0)


# ---------- Distance matrix ----------
def compute_distance_matrix(features: FrameFeatures,
                            cache_dir: Path,
                            cache_key: str,
                            weights: DistanceWeights = DistanceWeights()
                            ) -> np.ndarray:
    """N x N similarity matrix combining landmark L2 + pixel MSE + velocity diff."""
    cache_file = cache_dir / f"{cache_key}_distance.npy"
    if cache_file.is_file():
        LOG.info("Loading cached distance matrix from %s", cache_file)
        return np.load(cache_file)

    LOG.info("Computing distance matrix...")
    n = features.landmarks.shape[0]

    # NaN-safe landmark vector (missing face coords → 0 so they contribute
    # nothing rather than poisoning the row).
    lm = np.nan_to_num(features.landmarks, nan=0.0).reshape(n, -1)
    LOG.info("  · landmark L2   (%d × %d)", n, n)
    lm_dist  = _pairwise_l2(lm)

    LOG.info("  · pixel MSE     (%d × %d)", n, n)
    thumbs   = features.thumb_gray.astype(np.float32).reshape(n, -1) / 255.0
    pix_dist = _pairwise_mse(thumbs)

    LOG.info("  · velocity diff")
    v        = features.velocity_mag.astype(np.float32)
    vel_dist = np.abs(v[:, None] - v[None, :])

    w_sum = weights.landmark + weights.pixel + weights.velocity
    d = (
        weights.landmark * _normalize_robust(lm_dist)
        + weights.pixel * _normalize_robust(pix_dist)
        + weights.velocity * _normalize_robust(vel_dist)
    ) / max(w_sum, 1e-9)

    # Mask out non-loopable rows/cols + self-loops.
    invalid = ~features.valid_mask
    d[invalid, :] = np.inf
    d[:, invalid] = np.inf
    np.fill_diagonal(d, np.inf)

    cache_dir.mkdir(parents=True, exist_ok=True)
    np.save(cache_file, d.astype(np.float32))
    finite = d[np.isfinite(d)]
    LOG.info("Cached distance matrix → %s (min=%.4f, p10=%.4f)",
             cache_file,
             float(finite.min()) if finite.size else float("nan"),
             float(np.quantile(finite, 0.10)) if finite.size else float("nan"))
    return d.astype(np.float32)


# ---------- Transition graph ----------
def build_transition_graph(d: np.ndarray,
                           features: FrameFeatures,
                           threshold: float,
                           min_gap: int,
                           velocity_align: float = 0.6,
                           max_per_node: int = 32
                           ) -> Dict[int, List[Tuple[int, float]]]:
    """Convert the dense distance matrix into a sparse graph of valid jumps."""
    LOG.info("Building transition graph (threshold=%.4f, min_gap=%d)",
             threshold, min_gap)
    n        = d.shape[0]
    vel_vec  = features.velocity_vec
    vel_mag  = features.velocity_mag
    positive = vel_mag[vel_mag > 0]
    rest_thresh = float(np.quantile(positive, 0.10)) if positive.size else 0.0

    graph: Dict[int, List[Tuple[int, float]]] = {}
    edges = 0
    for i in range(n):
        if not features.valid_mask[i]:
            continue
        row = d[i]
        idx = np.where((row < threshold) & np.isfinite(row))[0]
        if idx.size == 0:
            continue
        kept: List[Tuple[int, float]] = []
        vi = vel_vec[i]
        mi = float(vel_mag[i])
        for j in idx:
            j_int = int(j)
            if abs(j_int - i) <= min_gap:
                continue
            if not features.valid_mask[j_int]:
                continue
            mj = float(vel_mag[j_int])
            both_rest = (mi <= rest_thresh) and (mj <= rest_thresh)
            cos = 1.0
            if mi > 1e-6 and mj > 1e-6:
                vj = vel_vec[j_int]
                denom = float(np.linalg.norm(vi) * np.linalg.norm(vj)) + 1e-9
                cos = float(np.dot(vi, vj) / denom)
            if not (both_rest or cos >= velocity_align):
                continue
            kept.append((j_int, float(row[j_int])))
        if kept:
            kept.sort(key=lambda t: t[1])
            graph[i] = kept[:max_per_node]
            edges += len(graph[i])

    LOG.info("Graph: %d source nodes · %d edges (avg %.1f per node)",
             len(graph), edges, edges / max(1, len(graph)))
    return graph


# ---------- Random walk ----------
def random_walk(transitions: Dict[int, List[Tuple[int, float]]],
                n_frames: int,
                target_frames: int,
                config: WalkConfig,
                rng: random.Random) -> List[Op]:
    """Walk the transition graph until target_frames have been planned."""
    ops: List[Op] = []
    cur = 0
    emitted = 0
    cooldown = 0
    recent: Deque[int] = deque(maxlen=config.recent_destinations)
    hard_cuts = 0

    progress = tqdm(total=target_frames, desc="Planning sequence", unit="frame")
    while emitted < target_frames:
        if cur >= n_frames:
            cur = 0
            hard_cuts += 1
            LOG.debug("Hard wrap to frame 0 (cut #%d)", hard_cuts)

        end_horizon = n_frames - (config.transition_len + 1)
        forced = cur > end_horizon
        can_jump = (
            cooldown <= 0
            and cur in transitions
            and cur + config.transition_len <= n_frames
        )

        if forced or (can_jump and rng.random() < config.p_jump):
            pool = transitions.get(cur, [])
            usable = [(j, dist) for (j, dist) in pool
                      if j + config.transition_len <= n_frames
                      and j not in recent]
            if not usable and forced and pool:
                # Near-EOF starvation: ignore quarantine before falling back
                # to a hard cut.
                recent.clear()
                usable = [(j, dist) for (j, dist) in pool
                          if j + config.transition_len <= n_frames]

            if usable:
                w = np.fromiter((1.0 / (d + 1e-6) for _, d in usable),
                                dtype=np.float64, count=len(usable))
                w /= w.sum()
                pick = rng.choices(range(len(usable)),
                                   weights=w.tolist(), k=1)[0]
                j, _ = usable[pick]
                ops.append(Op("transition", cur, j))
                emitted += config.transition_len
                progress.update(config.transition_len)
                recent.append(j)
                cur = j + config.transition_len
                cooldown = config.cooldown_frames
                continue
            elif forced:
                cur = n_frames  # force wrap on next iter
                continue

        ops.append(Op("linear", cur))
        emitted += 1
        progress.update(1)
        cur += 1
        cooldown = max(0, cooldown - 1)

    progress.close()
    if hard_cuts:
        LOG.warning("Sequencer required %d hard wrap(s). Consider raising "
                    "--threshold or lowering target duration.", hard_cuts)
    return ops


In [ ]:
# Cell 7 — Motion-compensated optical-flow blend + streaming ffmpeg renderer
# (writes a silent local mp4; uploading + cleanup happens in Cell 8)

def motion_compensated_blend(src: np.ndarray,
                             dst: np.ndarray,
                             alpha: float,
                             flow_calc) -> np.ndarray:
    """Bidirectional optical-flow warping + linear alpha blend.

    ``alpha=0`` returns ``src``-dominant output, ``alpha=1`` returns
    ``dst``-dominant output. Pixels are warped toward each other so that
    hair, fabric folds and finger positions morph across the transition
    instead of cross-dissolving in place.
    """
    if src.shape != dst.shape:
        dst = cv2.resize(dst, (src.shape[1], src.shape[0]))

    gs = cv2.cvtColor(src, cv2.COLOR_BGR2GRAY)
    gd = cv2.cvtColor(dst, cv2.COLOR_BGR2GRAY)
    flow_fwd = _calc_flow(flow_calc, gs, gd)
    flow_bwd = _calc_flow(flow_calc, gd, gs)

    h, w = src.shape[:2]
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    map_x_src = xx + flow_fwd[..., 0] * alpha
    map_y_src = yy + flow_fwd[..., 1] * alpha
    map_x_dst = xx + flow_bwd[..., 0] * (1.0 - alpha)
    map_y_dst = yy + flow_bwd[..., 1] * (1.0 - alpha)

    warped_src = cv2.remap(src, map_x_src, map_y_src,
                           interpolation=cv2.INTER_LINEAR,
                           borderMode=cv2.BORDER_REFLECT)
    warped_dst = cv2.remap(dst, map_x_dst, map_y_dst,
                           interpolation=cv2.INTER_LINEAR,
                           borderMode=cv2.BORDER_REFLECT)
    blended = ((1.0 - alpha) * warped_src.astype(np.float32)
               + alpha * warped_dst.astype(np.float32))
    return np.clip(blended, 0.0, 255.0).astype(np.uint8)


class FrameReader:
    """Random-access frame reader with sequential-read fast-path + LRU cache.

    OpenCV's seek is expensive on H.264 (decoder must rewind to the previous
    keyframe), so we keep an LRU of the last ``cache_size`` decoded frames.
    The renderer's access pattern is mostly sequential with occasional jumps,
    which this matches well.
    """

    def __init__(self, video_path: str, cache_size: int = 128) -> None:
        self.cap = cv2.VideoCapture(video_path)
        if not self.cap.isOpened():
            raise IOError(f"Could not open video: {video_path}")
        self.expected_next = 0
        self._cache: Dict[int, np.ndarray] = {}
        self._order: Deque[int] = deque()
        self._cache_size = cache_size

    def read(self, frame_idx: int) -> np.ndarray:
        if frame_idx in self._cache:
            return self._cache[frame_idx]
        if frame_idx != self.expected_next:
            self.cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            self.expected_next = frame_idx
        ok, frame = self.cap.read()
        if not ok or frame is None:
            raise IOError(f"Failed to read frame {frame_idx}")
        self.expected_next = frame_idx + 1
        self._cache[frame_idx] = frame
        self._order.append(frame_idx)
        while len(self._order) > self._cache_size:
            self._cache.pop(self._order.popleft(), None)
        return frame

    def close(self) -> None:
        self.cap.release()

    def __enter__(self) -> "FrameReader":
        return self

    def __exit__(self, exc_type, exc, tb) -> None:
        self.close()


def stream_frames(ops: List[Op],
                  reader: FrameReader,
                  transition_len: int) -> Generator[np.ndarray, None, None]:
    """Yield each output frame on demand — never buffer the whole video."""
    flow_calc = _make_dis_flow(cv2.DISOPTICAL_FLOW_PRESET_MEDIUM)
    for op in ops:
        if op.kind == "linear":
            yield reader.read(op.src)
        elif op.kind == "transition":
            i, j = op.src, op.dst
            denom = max(1, transition_len - 1)
            for t in range(transition_len):
                alpha = float(t) / float(denom)
                src = reader.read(i + t)
                dst = reader.read(j + t)
                yield motion_compensated_blend(src, dst, alpha, flow_calc)
        else:
            raise ValueError(f"Unknown op kind: {op.kind!r}")


def render_to_local_mp4(input_video: str,
                        output_video: str,
                        ops: List[Op],
                        meta: VideoMetadata,
                        total_out_frames: int,
                        transition_len: int) -> Path:
    """Pipe blended frames into a libx264 ffmpeg subprocess (silent)."""
    if shutil.which("ffmpeg") is None:
        raise RuntimeError(
            "ffmpeg not found on PATH. Install it (e.g. `brew install ffmpeg`)."
        )

    output_video = str(output_video)
    Path(output_video).parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        "ffmpeg", "-y",
        "-hide_banner", "-loglevel", "error",
        "-f", "rawvideo",
        "-vcodec", "rawvideo",
        "-pix_fmt", "bgr24",
        "-s", f"{meta.width}x{meta.height}",
        "-r", f"{meta.fps:.6f}",
        "-i", "-",
        "-an",                         # silent (no audio track)
        "-c:v", "libx264",
        "-pix_fmt", "yuv420p",
        "-preset", "medium",
        "-crf", "18",
        "-movflags", "+faststart",
        output_video,
    ]
    LOG.info("Spawning ffmpeg → %s", output_video)
    proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stderr=subprocess.PIPE)

    written = 0
    try:
        with FrameReader(input_video, cache_size=128) as reader, \
                tqdm(total=total_out_frames, desc="Rendering",
                     unit="frame") as bar:
            for frame in stream_frames(ops, reader, transition_len):
                if frame.shape[1] != meta.width or frame.shape[0] != meta.height:
                    frame = cv2.resize(frame, (meta.width, meta.height))
                proc.stdin.write(frame.tobytes())
                written += 1
                bar.update(1)
    except (BrokenPipeError, IOError) as exc:
        LOG.error("Pipe to ffmpeg broke after %d frames: %s", written, exc)
    finally:
        try:
            if proc.stdin:
                proc.stdin.close()
        except BrokenPipeError:
            pass
        stderr_bytes = proc.stderr.read() if proc.stderr else b""
        rc = proc.wait()
        if rc != 0:
            LOG.error("ffmpeg failed (rc=%d):\n%s",
                      rc, stderr_bytes.decode("utf-8", errors="replace"))
            raise RuntimeError("ffmpeg encoding failed")
    LOG.info("Rendered %d frames (~%.1fs) → %s",
             written, written / max(meta.fps, 1e-6), output_video)
    return Path(output_video)


In [ ]:
# Cell 8 — End-to-end pipeline: download → analyze → render → upload → cleanup

def run_pipeline(input_url: str = INPUT_S3_URL,
                 output_s3: str = OUTPUT_S3_FOLDER,
                 duration_sec: float = TARGET_DURATION_SEC,
                 threshold: float = SIMILARITY_THRESHOLD,
                 min_gap: int = MIN_GAP_FRAMES,
                 jump_probability: float = JUMP_PROBABILITY,
                 transition_len: int = TRANSITION_FRAMES,
                 cooldown: int = COOLDOWN_FRAMES,
                 velocity_align: float = VELOCITY_ALIGN,
                 seed: Optional[int] = RANDOM_SEED,
                 delete_local_after_upload: bool = DELETE_LOCAL_VIDEO_AFTER_UPLOAD,
                 ) -> Dict[str, object]:
    """Run the full Video Textures pipeline end-to-end with S3 I/O."""

    # 1. Pull the source clip from S3 / HTTP into the local cache.
    local_input = download_from_s3(input_url)

    # 2. MediaPipe + thumbnails + velocity (cached in .npz).
    meta = get_video_metadata(str(local_input))
    LOG.info("Input: %dx%d @ %.2f fps · %d frames (~%.1fs) · fourcc=%r",
             meta.width, meta.height, meta.fps, meta.frame_count,
             meta.frame_count / max(meta.fps, 1e-6), meta.fourcc)
    if meta.frame_count <= min_gap + transition_len + 2:
        raise ValueError(
            f"Input has only {meta.frame_count} frames — too short for "
            f"min_gap={min_gap}. Lower MIN_GAP_FRAMES or use a longer clip."
        )
    cache_key = video_cache_key(str(local_input))
    features  = extract_features(meta, CACHE_ROOT, cache_key)
    if not features.valid_mask.any():
        raise RuntimeError(
            "MediaPipe failed to detect a person in any frame; cannot proceed."
        )

    # 3. Distance matrix (cached in .npy) and transition graph.
    d_matrix = compute_distance_matrix(features, CACHE_ROOT, cache_key)
    graph = build_transition_graph(
        d_matrix, features,
        threshold=threshold, min_gap=min_gap,
        velocity_align=velocity_align,
    )
    if not graph:
        raise RuntimeError(
            f"No valid transitions at threshold={threshold}. "
            "Try raising SIMILARITY_THRESHOLD or lowering MIN_GAP_FRAMES."
        )

    # 4. Plan the random walk.
    rng = random.Random(seed) if seed is not None else random.Random()
    walk_cfg = WalkConfig(
        p_jump=jump_probability,
        transition_len=transition_len,
        cooldown_frames=cooldown,
    )
    target_frames = int(round(duration_sec * meta.fps))
    ops = random_walk(graph, meta.frame_count, target_frames, walk_cfg, rng)
    total_out = sum(transition_len if op.kind == "transition" else 1
                    for op in ops)
    n_jumps   = sum(1 for op in ops if op.kind == "transition")
    LOG.info("Plan: %d ops · %d jumps · %d frames (~%.1fs)",
             len(ops), n_jumps, total_out, total_out / meta.fps)

    # 5. Render locally with FFmpeg into the cache.
    local_output = RENDER_DIR / f"{cache_key}_{int(duration_sec)}s.mp4"
    render_to_local_mp4(
        str(local_input), str(local_output), ops, meta, total_out,
        transition_len=transition_len,
    )

    # 6. Upload to S3 with progress bar.
    final_uri = upload_to_s3(local_output, output_s3)

    # 7. Clean up heavy local mp4s; keep the .npz/.npy feature caches.
    if delete_local_after_upload:
        cleanup_local_videos(local_input, local_output)

    return {
        "input_url":        input_url,
        "local_input":      str(local_input),
        "local_output":     str(local_output),
        "s3_uri":           final_uri,
        "frames_rendered":  total_out,
        "duration_seconds": total_out / max(meta.fps, 1e-6),
        "fps":              meta.fps,
        "resolution":       (meta.width, meta.height),
        "num_jumps":        n_jumps,
    }


# Kick off the full pipeline. Re-running is cheap thanks to caching.
result = run_pipeline()
print("\nPipeline finished:")
for k, v in result.items():
    print(f"  {k:18s} = {v}")
